# Exploratory SuSiEx fits

In [1]:
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    
    # -----------------------------------------------------------------------------
    # Two susieR builds, each in its own environment so they never clobber.
    # Each susie_*() carries its enclosing env, so internal helper lookups resolve
    # against the build it came from.
    #   susie_hc : Haochen multi-panel / R_finite build -> vanilla + _pop (eb)
    #   susie_xw : Xuewei ld_mismatch build             -> _mix  (eb_mix)
    # -----------------------------------------------------------------------------


    # Local susieR build (multi-panel / R_finite) -- required for the list() reference
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

Loading required package: Rcpp

Loading required package: RcppEigen

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘MASS’


The following object is masked _by_ ‘.GlobalEnv’:

    select


The following object is masked from ‘package:dplyr’:

    select



Attaching package: ‘crayon’


The following object is masked from ‘package:ggplot2’:

    %+%



Attaching package: ‘pecotmr’


The following objects are masked _by_ ‘.GlobalEnv’:

    dentist, raiss, sdpr, slalom



Attaching package: ‘susieR’



In [2]:
   cross_cs_maxcorr <- function(fit, R) {
      if (is.null(fit)) return(NA_real_)
      cs <- fit$sets$cs
      if (is.null(cs) || length(cs) < 2) return(NA_real_)
      tops <- vapply(cs, function(m) m[which.max(fit$pip[m])], integer(1))
      cc <- abs(R[tops, tops]); diag(cc) <- NA
      max(cc, na.rm = TRUE)
    }

In [65]:
ld_path = "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/EUR_AFR_Mix_simulation/result/h2g_0.15/causal_3/prop_0.20//sample_0_ncausal_3_h2g_0.15_prop_0.2_mix_pop.rds"

In [66]:
    dats  <- readRDS(ld_path)
    n_eur <- dats$n_eur; n_afr <- dats$n_afr; N <- dats$total_n
    w     <- n_afr / N  

In [55]:
    EUR_sub <- get_cormat(dats$eur_Xmat)
    AFR_sub <- get_cormat(dats$afr_Xmat)
    if (!setequal(colnames(EUR_sub), colnames(AFR_sub)))
      save_dropped("EUR/AFR subsample variant sets differ")
    ref_order <- colnames(EUR_sub)
    AFR_sub   <- AFR_sub[ref_order, ref_order]

    ## ---- full-panel LD (realistic reference-panel case) ----
    panel    <- readRDS(dats$panel_ld_path)
    EUR_full <- panel$EUR_LD_full[ref_order, ref_order]
    AFR_full <- panel$AFR_LD_full[ref_order, ref_order]

    ## ---- natural-weight meta z (direct; both cohorts already aligned) ----
    e_idx <- match(ref_order, dats$eur_target_tb$variant_id)
    a_idx <- match(ref_order, dats$afr_target_tb$variant_id)
     if(nrow(dats$eur_target_tb) == 0){
         z_meta = dats$afr_target_tb$z
     }else if(nrow(dats$afr_target_tb) == 0){
         z_meta = dats$eur_target_tb$z
     }else{
        if (anyNA(e_idx) || anyNA(a_idx)) save_dropped("sumstat variants do not cover ref_order")
        z_eur  <- dats$eur_target_tb$z[e_idx]
        z_afr  <- dats$afr_target_tb$z[a_idx]
        z_meta <- (sqrt(n_afr) * z_afr + sqrt(n_eur) * z_eur) / sqrt(N)
    }
    n_mix  <- N                                # n_eff at the natural weight = total_n


    ## ---- generative LD both ways ----
    R_gen_sub  <- w * AFR_sub  + (1 - w) * EUR_sub     # GOLD STANDARD
    R_gen_full <- w * AFR_full + (1 - w) * EUR_full

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.



In [57]:
eur_fit$sets

,min.abs.corr,mean.abs.corr,median.abs.corr
,<dbl>,<dbl>,<dbl>
L1,1.0000000,1.0000000,1.0000000
L2,1.0000000,1.0000000,1.0000000
L3,0.9964779,0.9978867,0.9964779


In [58]:
eur_fit = susie_rss(z = dats$eur_target_tb$z, R = EUR_full, n = n_eur)

afr_fit = susie_rss(z = dats$afr_target_tb$z, R = AFR_full, n = n_afr)

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.



In [59]:
afr_fit$sets

$cs
NULL

$coverage
NULL

$requested_coverage
[1] 0.95

In [60]:
res = susie_post_outcome_configuration(list(eur_fit, afr_fit), method = "susiex")

Warning message:
“Skipping trait view(s) with no credible sets for post-hoc analysis: trait_2”
Fewer than two trait views with credible sets for post-hoc outcome configuration analysis; returning NULL.



In [52]:
/sc/arion/projects/ad-omics/haochen/rss_qc/susie_x/SuSiEx/bin 

$susiex
$susiex$config_1
$susiex$config_1$config_probability
  trait_1 trait_2  config_prob
1       0       0 1.994343e-09
2       1       0 7.977390e-01
3       0       1 2.970738e-08
4       1       1 2.022610e-01

$susiex$config_1$config_summary
        cs_indices logBF_outcome posthoc_prob active
trait_1         L3     19.806977     1.000000   TRUE
trait_2         L3      2.701081     0.202261  FALSE

attr(,"raw")
attr(,"raw")$cs_indices
trait_1 trait_2 
      3       3 

attr(,"raw")$cs_labels
trait_1 trait_2 
   "L3"    "L3" 

attr(,"raw")$logBF_trait
  trait_1   trait_2 
19.806977  2.701081 

attr(,"raw")$configs
     trait_1 trait_2
[1,]       0       0
[2,]       1       0
[3,]       0       1
[4,]       1       1

attr(,"raw")$config_prob
[1] 1.994343e-09 7.977390e-01 2.970738e-08 2.022610e-01

attr(,"raw")$marginal_prob
 trait_1  trait_2 
1.000000 0.202261 

attr(,"raw")$active
trait_1 trait_2 
   TRUE   FALSE 


$susiex$config_2
$susiex$config_2$config_probability
  trait_1 trait_2  config_prob
1       0       0 1.255018e-11
2       1       0 2.283180e-04
3       0       1 1.869452e-10
4       1       1 9.997717e-01

$susiex$config_2$config_summary
        cs_indices logBF_outcome posthoc_prob active
trait_1         L2     16.716515    1.0000000   TRUE
trait_2         L3      2.701081    0.9997717   TRUE

attr(,"raw")
attr(,"raw")$cs_indices
trait_1 trait_2 
      2       3 

attr(,"raw")$cs_labels
trait_1 trait_2 
   "L2"    "L3" 

attr(,"raw")$logBF_trait
  trait_1   trait_2 
16.716515  2.701081 

attr(,"raw")$configs
     trait_1 trait_2
[1,]       0       0
[2,]       1       0
[3,]       0       1
[4,]       1       1

attr(,"raw")$config_prob
[1] 1.255018e-11 2.283180e-04 1.869452e-10 9.997717e-01

attr(,"raw")$marginal_prob
  trait_1   trait_2 
1.0000000 0.9997717 

attr(,"raw")$active
trait_1 trait_2 
   TRUE    TRUE 


$susiex$config_3
$susiex$config_3$config_probability
  trait_1 trait_2  config_prob
1       0       0 3.634548e-09
2       1       0 8.137903e-01
3       0       1 5.413957e-08
4       1       1 1.862096e-01

$susiex$config_3$config_summary
        cs_indices logBF_outcome posthoc_prob active
trait_1         L1     19.226729    0.9999999   TRUE
trait_2         L3      2.701081    0.1862097  FALSE

attr(,"raw")
attr(,"raw")$cs_indices
trait_1 trait_2 
      1       3 

attr(,"raw")$cs_labels
trait_1 trait_2 
   "L1"    "L3" 

attr(,"raw")$logBF_trait
  trait_1   trait_2 
19.226729  2.701081 

attr(,"raw")$configs
     trait_1 trait_2
[1,]       0       0
[2,]       1       0
[3,]       0       1
[4,]       1       1

attr(,"raw")$config_prob
[1] 3.634548e-09 8.137903e-01 5.413957e-08 1.862096e-01

attr(,"raw")$marginal_prob
  trait_1   trait_2 
0.9999999 0.1862097 

attr(,"raw")$active
trait_1 trait_2 
   TRUE   FALSE 


$susiex$config_4
$susiex$config_4$config_probability
  trait_1 trait_2  config_prob
1       0       0 2.094946e-09
2       1       0 8.379805e-01
3       0       1 6.672920e-06
4       1       1 1.620129e-01

$susiex$config_4$config_summary
        cs_indices logBF_outcome posthoc_prob active
trait_1         L3     19.806977    0.9999933   TRUE
trait_2         L2      8.066285    0.1620195  FALSE

attr(,"raw")
attr(,"raw")$cs_indices
trait_1 trait_2 
      3       2 

attr(,"raw")$cs_labels
trait_1 trait_2 
   "L3"    "L2" 

attr(,"raw")$logBF_trait
  trait_1   trait_2 
19.806977  8.066285 

attr(,"raw")$configs
     trait_1 trait_2
[1,]       0       0
[2,]       1       0
[3,]       0       1
[4,]       1       1

attr(,"raw")$config_prob
[1] 2.094946e-09 8.379805e-01 6.672920e-06 1.620129e-01

attr(,"raw")$marginal_prob
  trait_1   trait_2 
0.9999933 0.1620195 

attr(,"raw")$active
trait_1 trait_2 
   TRUE   FALSE 


$susiex$config_5
$susiex$config_5$config_probability
  trait_1 trait_2  config_prob
1       0       0 4.630934e-08
2       1       0 8.424786e-01
3       0       1 1.475067e-04
4       1       1 1.573739e-01

$susiex$config_5$config_summary
        cs_indices 

In [69]:
# =====================================================================
# run_susiex(): drive the SuSiEx binary directly from in-memory
# Z-scores + LD matrices (the same objects you pass to susie_rss()).
#
# Why this works with Z-scores only:
#   SuSiEx reads BETA and SE columns, but internally it only ever uses
#   the ratio eff/se (= Z) together with the per-population sample size N
#   (see data.cpp:  cur.beta = eff/se/sqrt(N) ; model.cpp: v_sq = sigma^2/N).
#   So we put Z in the BETA column and set SE = 1.  bit-for-bit the same
#   standardized effect (Z/sqrt(N)) the model would get from real beta/se.
#
#   It also normally computes LD from a PLINK panel, but if the precomputed
#   files (<prefix>.ld.bin / _ref.bim / _frq.frq) already exist it skips
#   that step entirely (main.cpp: par.precmp).  We write those three files
#   ourselves, so no reference panel / PLINK is needed.
#
# Allele bookkeeping (so NO sign flip happens and R is used verbatim):
#   - SuSiEx never sign-flips the LD matrix (the ldflag line in data.cpp is
#     commented out) -> R goes in exactly as given.
#   - It DOES flip beta if sumstats alleles don't match the reference in the
#     orientation coord.a1==A1 & coord.a2==A2. We build the .bim/.frq so that
#     no flip occurs, i.e. beta (=Z) is kept as-is, matching R.
#   Requirement: your R must be the LD *correlation* matrix in the same allele
#   orientation as z (i.e. correlations of the a1-allele dosage) -- exactly
#   what susie_rss(z=z, R=R) assumes.
# =====================================================================

run_susiex <- function(pops,
                       susiex_bin,
                       workdir     = tempfile("susiex_"),
                       out_name    = "susiex_run",
                       chr         = "1",
                       n_sig       = 10,
                       level       = 0.95,
                       min_purity  = 0.5,
                       pval_thresh = 1e-5,
                       maf         = 0,
                       mult_step   = TRUE,
                       threads     = 1,
                       plink       = "plink",   # required by the arg parser but NOT executed
                                                # when precomputed LD files exist (precmp mode)
                       keep_ambig  = TRUE,
                       cleanup     = FALSE,
                       verbose     = TRUE) {

  ## ---- pops: a named list; each element describes one population ----
  ## Required per population:
  ##   z   numeric[m]           Z-scores, aligned to the effect allele a1
  ##   R   matrix[m,m]          LD correlation matrix, SAME order as z, a1 orientation
  ##   n   numeric(1)           GWAS sample size
  ## Optional per population (defaults generated if missing):
  ##   snp character[m]         variant IDs  (MUST be identical across pops for the
  ##                            same variant so they get merged; default "snp<bp>")
  ##   bp  integer[m]           base-pair positions (shared across pops per variant)
  ##   a1  character[m]         effect allele  (default "A")
  ##   a2  character[m]         other allele   (default "G")

  stopifnot(is.list(pops), length(pops) >= 1, !is.null(names(pops)))
  dir.create(workdir, showWarnings = FALSE, recursive = TRUE)

  prefixes <- character(length(pops))
  sst_files <- character(length(pops))
  n_gwas   <- numeric(length(pops))

  for (i in seq_along(pops)) {
    p   <- pops[[i]]
    nm  <- names(pops)[i]
    z   <- as.numeric(p$z)
    R   <- as.matrix(p$R)
    m   <- length(z)
    if (nrow(R) != m || ncol(R) != m)
      stop(sprintf("pop '%s': R is %dx%d but z has length %d", nm, nrow(R), ncol(R), m))

    bp  <- if (!is.null(p$bp))  as.integer(p$bp) else seq_len(m)
    snp <- if (!is.null(p$snp)) as.character(p$snp) else paste0("snp", bp)
    a1  <- if (!is.null(p$a1))  as.character(p$a1) else rep("A", m)  # effect allele (z orientation)
    a2  <- if (!is.null(p$a2))  as.character(p$a2) else rep("G", m)

    # keep everything in one consistent row order (also the LD .bin order)
    ord <- seq_len(m)
    R   <- R[ord, ord, drop = FALSE]
    diag(R) <- 1                                   # SuSiEx forces diagonal = 1 anyway

    pref <- file.path(workdir, nm)
    prefixes[i] <- pref
    n_gwas[i]   <- p$n

    # --- 1) summary statistics: BETA := Z, SE := 1, P from Z -----------
    pval <- 2 * pnorm(-abs(z))
    sst  <- data.frame(chr = chr, snp = snp, bp = bp,
                       A1 = a1, A2 = a2,              # A1 = effect allele
                       BETA = z, SE = 1, P = pval,
                       stringsAsFactors = FALSE)
    sst_files[i] <- paste0(pref, ".sumstats.txt")
    write.table(sst, sst_files[i], sep = "\t", quote = FALSE,
                row.names = FALSE, col.names = TRUE)

    # --- 2) _ref.bim : col5 = a2 , col6 = a1 (no-flip orientation) ------
    #     SuSiEx: coord.a2 <- bim col5 ; coord.a1 <- bim col6
    bim <- data.frame(chr = chr, snp = snp, cm = 0, bp = bp,
                      A1_bim = a2, A2_bim = a1, stringsAsFactors = FALSE)
    write.table(bim, paste0(pref, "_ref.bim"), sep = "\t", quote = FALSE,
                row.names = FALSE, col.names = FALSE)

    # --- 3) _frq.frq : col3 = a2 , col4 = a1 (matches bim) --------------
    #     SuSiEx: frq col3 -> a2 , frq col4 -> a1 ; must equal the bim coords
    frq <- data.frame(CHR = chr, SNP = snp, A1 = a2, A2 = a1,
                      MAF = 0.3, NCHROBS = round(2 * p$n),
                      stringsAsFactors = FALSE)
    write.table(frq, paste0(pref, "_frq.frq"), sep = "\t", quote = FALSE,
                row.names = FALSE, col.names = TRUE)

    # --- 4) .ld.bin : nvar*nvar float32, row-major (symmetric => order ok)
    con <- file(paste0(pref, ".ld.bin"), "wb")
    writeBin(as.numeric(as.vector(R)), con, size = 4)   # single precision
    close(con)
  }

  ## ---- column numbers in the sumstats file (fixed by our writer) ----
  ncol_str  <- function(v) paste(rep(v, length(pops)), collapse = ",")
  bp_all    <- unlist(lapply(pops, function(p) if (!is.null(p$bp)) p$bp else seq_along(p$z)))
  bp_start  <- min(bp_all); bp_end <- max(bp_all)

  args <- c(
    paste0("--sst_file=",    paste(sst_files, collapse = ",")),
    paste0("--n_gwas=",      paste(n_gwas, collapse = ",")),
    paste0("--ld_file=",     paste(prefixes, collapse = ",")),
    paste0("--out_dir=",     workdir),
    paste0("--out_name=",    out_name),
    paste0("--chr=",         chr),
    paste0("--bp=",          bp_start, ",", bp_end),
    paste0("--chr_col=",     ncol_str(1)),
    paste0("--snp_col=",     ncol_str(2)),
    paste0("--bp_col=",      ncol_str(3)),
    paste0("--a1_col=",      ncol_str(4)),
    paste0("--a2_col=",      ncol_str(5)),
    paste0("--eff_col=",     ncol_str(6)),
    paste0("--se_col=",      ncol_str(7)),
    paste0("--pval_col=",    ncol_str(8)),
    paste0("--n_sig=",       n_sig),
    paste0("--level=",       level),
    paste0("--min_purity=",  min_purity),
    paste0("--pval_thresh=", pval_thresh),
    paste0("--maf=",         maf),
    paste0("--mult-step=",   ifelse(mult_step, "True", "False")),
    paste0("--keep-ambig=",  ifelse(keep_ambig, "True", "False")),
    paste0("--plink=",       plink),
    paste0("--threads=",     threads)
  )

  if (verbose) message("Running: ", susiex_bin, " \\\n  ", paste(args, collapse = " \\\n  "))
  log <- suppressWarnings(system2(susiex_bin, args, stdout = TRUE, stderr = TRUE))
  status <- attr(log, "status")

  ## ---- parse the .summary output ----
  summ_path <- file.path(workdir, paste0(out_name, ".summary"))
  cs_path   <- file.path(workdir, paste0(out_name, ".cs"))
  n_cs <- NA_integer_; summary_tab <- NULL; cs_tab <- NULL
  if (file.exists(summ_path)) {
    raw <- readLines(summ_path)
    body <- raw[!grepl("^#", raw)]
    body <- body[nzchar(trimws(body))]
    if (length(body) && grepl("^(NULL|FAIL)", body[1])) {
      n_cs <- 0L
    } else if (length(body) >= 1) {
      summary_tab <- utils::read.table(text = paste(body, collapse = "\n"),
                                       header = TRUE, sep = "\t",
                                       stringsAsFactors = FALSE, check.names = FALSE)
      n_cs <- nrow(summary_tab)
    }
  }
  if (file.exists(cs_path)) {
    craw <- readLines(cs_path); cbody <- craw[!grepl("^#", craw)]
    cbody <- cbody[nzchar(trimws(cbody))]
    if (length(cbody) && !grepl("^(NULL|FAIL)", cbody[1]))
      cs_tab <- try(utils::read.table(text = paste(cbody, collapse = "\n"),
                                      header = TRUE, sep = "\t",
                                      stringsAsFactors = FALSE, check.names = FALSE),
                    silent = TRUE)
  }

  if (cleanup) unlink(workdir, recursive = TRUE)

  list(n_cs = n_cs, summary = summary_tab, cs = cs_tab,
       log = log, status = status, workdir = workdir,
       summary_path = summ_path, cs_path = cs_path)
}

In [70]:
# =====================================================================
# run_susiex_joint.R  --  restructure your existing SuSiE inputs into
# SuSiEx precomputed-LD inputs and run YOUR installed SuSiEx binary.
#
# Run this in the SAME R session where these objects already exist:
#   dats$eur_target_tb$z , EUR_full , n_eur
#   dats$afr_target_tb$z , AFR_full , n_afr
#
# Usage:   source("run_susiex_joint.R")
# It prints the number of credible sets and the per-CS summary, and
# leaves the full SuSiEx output files in OUT_DIR below.
# =====================================================================

## ------------------------- EDIT IF NEEDED ---------------------------
SUSIEX_BIN <- "/sc/arion/projects/ad-omics/haochen/rss_qc/susie_x/SuSiEx/bin/SuSiEx"
OUT_DIR    <- "susiex_out"        # where inputs + results are written
OUT_NAME   <- "EUR_AFR"           # output file prefix
CHR        <- "1"                 # nominal chromosome label (any consistent value)
N_SIG      <- 10                  # max number of signals (L)
LEVEL      <- 0.95                # credible set coverage
MIN_PURITY <- 0.5                 # SuSiEx default
PVAL_THRESH<- 1e-5                # SuSiEx default marginal-p filter
MULT_STEP  <- TRUE               # multi-step fitting (recommended)
THREADS    <- 1
## --------------------------------------------------------------------

## Your inputs, referenced by the names you already use:
eur <- list(tb = dats$eur_target_tb, R = EUR_full, n = n_eur, name = "EUR")
afr <- list(tb = dats$afr_target_tb, R = AFR_full, n = n_afr, name = "AFR")

# ---- helper: pull metadata columns from a target table --------------
pick_col <- function(tb, candidates) {
  hit <- intersect(candidates, colnames(tb))
  if (length(hit)) tb[[hit[1]]] else NULL
}
meta <- function(tb, m, other_bp = NULL) {
  snp <- pick_col(tb, c("snp","SNP","rsid","rsID","variant","variant_id","id","ID","snps"))
  bp  <- pick_col(tb, c("pos","bp","position","BP","POS","pos_hg38","pos_hg19"))
  a1  <- pick_col(tb, c("a1","A1","alt","ALT","effect_allele","EA","ea"))
  a2  <- pick_col(tb, c("a2","A2","ref","REF","other_allele","NEA","nea"))
  if (is.null(bp))  bp  <- if (!is.null(other_bp)) other_bp else seq_len(m)
  if (is.null(snp)) snp <- paste0("v", bp)
  if (is.null(a1))  a1  <- rep("A", m)
  if (is.null(a2))  a2  <- rep("G", m)
  list(snp = as.character(snp), bp = as.integer(bp),
       a1 = as.character(a1), a2 = as.character(a2))
}

# ---- write the 4 files SuSiEx needs for one population --------------
write_pop <- function(p, dir) {
  z <- as.numeric(p$tb$z); m <- length(z)
  R <- as.matrix(p$R)
  if (nrow(R) != m || ncol(R) != m)
    stop(sprintf("%s: R is %dx%d but z has length %d", p$name, nrow(R), ncol(R), m))
  diag(R) <- 1
  md <- p$meta
  pref <- file.path(dir, p$name)

  # (1) sumstats: BETA := Z, SE := 1  (SuSiEx uses only eff/se = Z and N)
  sst <- data.frame(chr = CHR, snp = md$snp, bp = md$bp,
                    A1 = md$a1, A2 = md$a2,           # A1 = effect allele (z orientation)
                    BETA = z, SE = 1, P = 2 * pnorm(-abs(z)),
                    stringsAsFactors = FALSE)
  write.table(sst, paste0(pref, ".sumstats.txt"), sep = "\t",
              quote = FALSE, row.names = FALSE, col.names = TRUE)

  # (2) _ref.bim : col5 = a2, col6 = a1  -> no beta sign-flip in SuSiEx
  bim <- data.frame(CHR, md$snp, 0, md$bp, md$a2, md$a1, stringsAsFactors = FALSE)
  write.table(bim, paste0(pref, "_ref.bim"), sep = "\t",
              quote = FALSE, row.names = FALSE, col.names = FALSE)

  # (3) _frq.frq : col3 = a2, col4 = a1  (must match the .bim)
  frq <- data.frame(CHR = CHR, SNP = md$snp, A1 = md$a2, A2 = md$a1,
                    MAF = 0.3, NCHROBS = round(2 * p$n), stringsAsFactors = FALSE)
  write.table(frq, paste0(pref, "_frq.frq"), sep = "\t",
              quote = FALSE, row.names = FALSE, col.names = TRUE)

  # (4) .ld.bin : m*m float32, verbatim (SuSiEx never sign-flips LD)
  con <- file(paste0(pref, ".ld.bin"), "wb")
  writeBin(as.numeric(as.vector(R)), con, size = 4); close(con)

  pref
}

# ---- build inputs ---------------------------------------------------
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)
eur$meta <- meta(eur$tb, length(eur$tb$z))
afr$meta <- meta(afr$tb, length(afr$tb$z), other_bp = NULL)

cat("Detected metadata (verify these look right):\n")
cat(sprintf("  EUR: %d SNPs | id e.g. %s | bp e.g. %s | alleles %s/%s\n",
            length(eur$meta$snp), eur$meta$snp[1], eur$meta$bp[1], eur$meta$a1[1], eur$meta$a2[1]))
cat(sprintf("  AFR: %d SNPs | id e.g. %s | bp e.g. %s | alleles %s/%s\n",
            length(afr$meta$snp), afr$meta$snp[1], afr$meta$bp[1], afr$meta$a1[1], afr$meta$a2[1]))
shared <- length(intersect(eur$meta$snp, afr$meta$snp))
cat(sprintf("  shared variant IDs across EUR & AFR: %d\n", shared))
if (shared == 0)
  warning("No shared variant IDs -> SuSiEx will treat them as ancestry-specific. ",
          "Make sure both tables carry a common snp/pos key (they are matched by pos+ID).")

pref_eur <- write_pop(eur, OUT_DIR)
pref_afr <- write_pop(afr, OUT_DIR)

# ---- assemble the SuSiEx command -----------------------------------
all_bp <- c(eur$meta$bp, afr$meta$bp)
cc <- function(v, k = 2) paste(rep(v, k), collapse = ",")
args <- c(
  paste0("--sst_file=",   pref_eur, ".sumstats.txt,", pref_afr, ".sumstats.txt"),
  paste0("--n_gwas=",     eur$n, ",", afr$n),
  paste0("--ld_file=",    pref_eur, ",", pref_afr),
  paste0("--out_dir=",    OUT_DIR),
  paste0("--out_name=",   OUT_NAME),
  paste0("--chr=",        CHR),
  paste0("--bp=",         min(all_bp), ",", max(all_bp)),
  "--chr_col=1,1", "--snp_col=2,2", "--bp_col=3,3",
  "--a1_col=4,4", "--a2_col=5,5", "--eff_col=6,6", "--se_col=7,7", "--pval_col=8,8",
  paste0("--n_sig=",      N_SIG),
  paste0("--level=",      LEVEL),
  paste0("--min_purity=", MIN_PURITY),
  paste0("--pval_thresh=",PVAL_THRESH),
  "--maf=0",
  paste0("--mult-step=",  ifelse(MULT_STEP, "True", "False")),
  "--keep-ambig=True",
  "--plink=plink",           # required by the parser but NOT executed (precomputed-LD mode)
  paste0("--threads=",    THREADS)
)

cat("\nRunning SuSiEx:\n  ", SUSIEX_BIN, "\n", sep = "")
log <- suppressWarnings(system2(SUSIEX_BIN, args, stdout = TRUE, stderr = TRUE))
cat(paste(log, collapse = "\n"), "\n")

# ---- read + print the result ---------------------------------------
summ <- file.path(OUT_DIR, paste0(OUT_NAME, ".summary"))
cat("\n========================  RESULT  ========================\n")
if (file.exists(summ)) {
  raw  <- readLines(summ)
  body <- raw[!grepl("^#", raw)]; body <- body[nzchar(trimws(body))]
  if (!length(body) || grepl("^(NULL|FAIL)", body[1])) {
    cat("Credible sets: 0  (", ifelse(length(body), body[1], "no output"), ")\n", sep="")
  } else {
    tab <- read.table(text = paste(body, collapse = "\n"), header = TRUE,
                      sep = "\t", check.names = FALSE, stringsAsFactors = FALSE)
    cat("Credible sets:", nrow(tab), "\n\n")
    print(tab)
  }
  cat("\nFull output: ", normalizePath(summ), " (+ .cs / .snp)\n", sep = "")
} else {
  cat("No .summary file produced -- check the SuSiEx log above.\n")
}

Detected metadata (verify these look right):
  EUR: 857 SNPs | id e.g. chr10:118978164:T:C | bp e.g. 118978164 | alleles C/T
  AFR: 857 SNPs | id e.g. chr10:118978164:T:C | bp e.g. 118978164 | alleles C/T
  shared variant IDs across EUR & AFR: 857

Running SuSiEx:
  /sc/arion/projects/ad-omics/haochen/rss_qc/susie_x/SuSiEx/bin/SuSiEx

Software parameters:
--sst_file = susiex_out/EUR.sumstats.txt,susiex_out/AFR.sumstats.txt
--ld_file = susiex_out/EUR,susiex_out/AFR
--n_gwas = 360,90
--out_dir = susiex_out
--out_name = EUR_AFR
--chr = 1
--bp = 118978164,120142638
--chr_col = 1,1
--snp_col = 2,2
--bp_col = 3,3
--a1_col = 4,4
--a2_col = 5,5
--eff_col = 6,6
--se_col = 7,7
--pval_col = 8,8
--plink = plink
--keep-ambig = true
--mult-step = true
--precmp = true
--maf = 0
--level = 0.95
--min_purity = 0.5
--pval_thresh = 1e-05
--tol = 0.0001
--n_sig = 10
--max_iter = 100
--threads = 1
... parse reference file: susiex_out/EUR_ref.bim ...
... 857 SNPs in the fine-mapping region read from susiex_o

In [64]:
n_eur

[1] 20000